# Can I report diffusion, and is my selected window stable?
Run this notebook from top to bottom. Read the short result tables first; detailed diagnostics are optional.

1. **Possible reaction?** Stop if molecular membership may have changed.
2. **Stable window?** Check whether measured properties and molecular structure change with time.
3. **Measurable diffusion?** Check whether molecular displacement grows steadily, and whether different fits and time blocks agree.

A simulation can be stable but too short to measure diffusion. Conversely, a positive fitted diffusion coefficient does not establish equilibrium.

**First run:** set the directory, units and timestep. Keep the whole trajectory. Inspect the histories, then enter explicit later frame windows if needed. Do not tune settings merely to obtain a pass.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from MartiniFuncs.trajectory_analysis import analyze_run, select_frame_window
from MartiniFuncs.postprocessing import load_analysis_cache
from MartiniFuncs.diffusion_diagnostics import diffusion_diagnostics, scalar_stationarity
from MartiniFuncs.stability import get_molecular_com_rdf
from MartiniFuncs.reaction_detection import PossibleReactionError
import MartiniFuncs.thermo as thermo_io


In [ ]:
RUN_DIR = Path(r"/path/to/simulation")
CACHE_DIR = None  # Optional Path to martini-postprocess output/cache; skips reading the dump.
INPUT_FILE = DUMP_FILE = DATA_FILE = None
TYPE_SYMBOLS = None
UNITS = "metal"
TIMESTEP_PS = 0.0005  # per integration step
REACTION_ATOM_COUNTS = (11, 23)  # CPD + current MAL; use your actual molecular sizes
BOND_SCALE = 1.3
FRAME_STRIDE = 1
MAX_FRAMES = None  # preview only; cannot establish long-time convergence

# Original dump indices, zero-based, start inclusive / end exclusive.
# Add explicit later windows, e.g. [(0, None), (1000, 2000)].
WINDOWS = [(0, None)]
FIT_INTERVALS_PS = [(10., 30.), (20., 40.), (30., 50.)]  # lag intervals, not trajectory windows
BLOCK_DURATIONS_PS = [250., 500.]  # compare at least two block sizes
MIN_BLOCKS = 8
BOOTSTRAP_RESAMPLES = 2000
SEED = 1234
ALPHA_BOUNDS = (0.8, 1.2)
RELATIVE_TOLERANCE = 0.25
MAX_RELATIVE_CI = 0.5

# Absolute tolerances in each column's units. None reports diagnostics without
# claiming equivalence. Set these based on scientifically meaningful changes.
SCALAR_TOLERANCES = {"density_g_cm3": None, "volume_angstrom3": None}
# Optional CSV you produce from thermo output: unique 'frame' column (source
# index), plus e.g. potential_energy_eV, pressure_bar, temperature_K.
# Use the same frame/time mapping if supplying a manual CSV instead of .out/.log.
LOAD_THERMO = True  # discover a unique LAMMPS .out/.log automatically
THERMO_FILE = None  # optional explicit relative or absolute output filename
THERMO_BLOCK = None  # choose a zero-based table if repeated timesteps are ambiguous
THERMO_CSV = None  # alternative manual frame-indexed data; bypasses automatic output loading
RDF_BINS = 50
STRUCTURE_STRIDE = 10  # sample actual geometry for RDFs, unrelated to TOR
SHOW_DETAILS = False  # full numerical tables and block-by-block diffusion plots
WRITE_FILES = False
OUTPUT_DIR = Path("diffusion_assessment")


## 1. Load the trajectory and check for possible reactions
The settings above use **source frame numbers**, not LAMMPS timesteps. A window `(1000, 2000)` includes frames 1000 through 1999.

The check looks for a connected 34-atom component (11 CPD + 23 current MAL atoms). If found, the notebook stops and displays its identity. Inspect it before treating the trajectory as an unchanged molecular mixture. This size criterion flags possible products, not confirmed reactions.


**Cost:** every source frame is checked for possible products, even when plotting every Nth frame. Loading can take time. Once loaded, rerun only the cells below when changing analysis windows or fit settings. A preview covers only frames actually read.


### Configuration: mathematical meaning of the controls
Here $T$ is the number of selected **retained** frames, $\Delta t$ their spacing in ps, and $M$ the number of molecules of one species. Scientific screening thresholds are chosen heuristics, not established universal limits.

| Setting | Exact role and effect |
|---|---|
| `REACTION_ATOM_COUNTS=(a,b)` | Stop when a current connected component has $a+b$ atoms; 11+23=34 here. Counts, not distance thresholds. |
| `BOND_SCALE` | Multiplies the covalent-radius sum in the bond inequality below; affects molecular identity and possible-product flags. |
| `WINDOWS`, `FRAME_STRIDE`, `MAX_FRAMES`, `TIMESTEP_PS` | Define retained samples and time spacing as described below. A time window is not a lag interval. |
| `FIT_INTERVALS_PS` | Inclusive lag limits $[\tau_a,\tau_b]$ for separate free-intercept fits; at least four sampled positive lags per fit and three distinct intervals for sensitivity. |
| `BLOCK_DURATIONS_PS=d` | For diffusion, $b=\max(3,\lceil d/\Delta t\rceil+1)$ samples per block; actual span $(b-1)\Delta t$. Different from the scalar block rule. |
| `MIN_BLOCKS` | Require $B=\lfloor T/b\rfloor$ at least this large for diffusion (default 8). Each fit also requires $(b-1)\Delta t\ge5\tau_b$. |
| `BOOTSTRAP_RESAMPLES=R` | Number of replacement resamples of whole blocks (default 2000), not new MD samples. Larger $R$ stabilizes percentile estimates but cannot add physical information. |
| `SEED` | Seeds bootstrap sampling for reproducible intervals; does not affect MD or physical uncertainty. |
| `ALPHA_BOUNDS` | Inclusive acceptable range for the fitted exponent in $\log\mathrm{MSD}=c+\alpha\log\tau$ (default 0.8–1.2). The local plotted derivative is different. |
| `RELATIVE_TOLERANCE=\epsilon` | Maximum relative fit-interval spread, earlier/later block shift, and pooled/block discrepancy; three definitions below (default 0.25). |
| `MAX_RELATIVE_CI=\eta` | Maximum CI half-width divided by $\lvert\bar D\rvert$ (default 0.5). This is relative precision, not the confidence level. |
| `SCALAR_TOLERANCES[name]=\delta` | Maximum acceptable earlier/later scalar change in that column's units. The entire change CI must lie within $[-\delta,\delta]$; `None` makes no equivalence claim. |
| `RDF_BINS`, `STRUCTURE_STRIDE` | Number of radial bins and subsampling within each structural section. They affect RDF resolution/sampling, not the diffusion lag grid. |
| `THERMO_CSV` | Joins observables on exact source `frame`; no interpolation, time conversion or log parsing is performed. |
| `CACHE_DIR` | Reuses original cached physics/sampling settings. Editing loading settings does not modify cached arrays. Unscreened reaction caches are rejected. |
| `SHOW_DETAILS`, `WRITE_FILES`, paths | Presentation/persistence only; do not change estimators. |

Example: with $\Delta t=0.5$ ps, a 250 ps block has $b=501$ samples. Eight blocks require 4008 retained frames, spanning 2003.5 ps. The general sampling-budget entries are $T_{\rm required}=B_{\min}b$ and $t_{\rm required}=(T_{\rm required}-1)\Delta t$. A longer bootstrap does not reduce this physical sampling requirement.


### Mathematics: frames, time, units and molecular identity
Let $f$ be the zero-based source frame, $s_f$ its LAMMPS timestep, and $h=$ `TIMESTEP_PS` (ps per MD step). The displayed time is
$$t_f=s_fh.$$
`FRAME_INDEX` / `FRAME_INDICES` select $f$, not $s_f$. Where present, `FRAME_STRIDE` $q$ retains $f\bmod q=0$; `MAX_FRAMES` $F$ reads only $0\le f<F$. `STARTFRAME`, `ENDFRAME` or each `WINDOWS` entry $(a,b)$ keeps retained frames with $a\le f<b$; `None` means the available end. The lag spacing is $\Delta t=t_{j+1}-t_j$ between **retained** samples, not $h$. Paths and write/display switches do not enter these equations.

`UNITS` controls conversion on reading. In these workflows coordinates are Å, masses are atomic mass units, source forces are eV/Å, and reported velocities are Å/ps. `TYPE_SYMBOLS` sets the type-to-element mapping used to infer atomic properties when needed; an incorrect mapping can change masses and bond radii.

Where `BOND_SCALE` $s$ is used (default 1.3), atoms are connected when their periodic-image separation satisfies
$$\|\mathbf r_j+\mathbf nH-\mathbf r_i\|\le s(r_i^{\rm cov}+r_j^{\rm cov}).$$
$H$ contains cell vectors as rows, $\mathbf n$ is an integer image shift, and $r^{\rm cov}$ is the element's covalent radius (Å). Molecules are connected components of this inferred graph; increasing $s$ can merge close molecules. Persistent atom IDs are labels, not coordinate-array row indices.


### LAMMPS thermo output: matching and units
`LOAD_THERMO=True` discovers a unique one-line `Step` table in `.out`/`.log` files. `THERMO_FILE` resolves filename ambiguity; `THERMO_BLOCK` selects a header occurrence if timesteps repeat. `THERMO_CSV` remains an alternative manual input.

The join is exactly $s_{\rm dump}=s_{\rm thermo}$, never nearest-time matching. For this continuation, dump spacing is $1000h=0.5$ ps and thermo spacing is $2000h=1$ ps. Unmatched frame cells are missing; scalar calculations use the 1 ps thermo samples, while diffusion still uses the 0.5 ps COM samples. Each observable's sample count and spacing are printed. The code rejects repeated step keys and inconsistent printed Time rather than guessing a run segment.

For `metal`: Temp is K, Press and Pij are bar, energy is eV, Volume is Å³ and Density is g/cm³. $P_{\rm GPa}=P_{\rm bar}/10^4$. Values are copied, not reconstructed. Energy is **as printed**, including any `thermo_modify norm`; confirm whether it is total or per-atom before comparing absolute energies. The reported temperature inherits the LAMMPS temperature compute and its DOF/bias treatment. Prefixed thermo density/volume preserve independent comparisons with dump-derived values.

Older results without a `timestep` column recover exact step integers from the source dump headers, using recorded source `frame` indices. No conversion from rounded time is used; the coverage report states whether recovery occurred.


In [ ]:
# Refresh this stateless I/O module at the point of use so an already-open
# notebook cannot retain an older attach_thermo signature after a package update.
import importlib
import inspect
import MartiniFuncs.thermo as thermo_io
thermo_io = importlib.reload(thermo_io)
if "dump_file" not in inspect.signature(thermo_io.attach_thermo).parameters:
    raise RuntimeError(f"Outdated MartiniFuncs installation loaded from {thermo_io.__file__}; use this repository's environment.")
print("Thermo parser:", thermo_io.__file__)
try:
    if CACHE_DIR is None:
        result = analyze_run(
            RUN_DIR, input_file=INPUT_FILE, dump_file=DUMP_FILE, data_file=DATA_FILE,
            units=UNITS, timestep_ps=TIMESTEP_PS, type_symbols=TYPE_SYMBOLS,
            frame_stride=FRAME_STRIDE, max_frames=MAX_FRAMES,
            compute_pairs=False, degrees_of_freedom=None, progress=print,
            reaction_atom_counts=REACTION_ATOM_COUNTS, bond_scale=BOND_SCALE,
        )
    else:
        result, cache_manifest = load_analysis_cache(CACHE_DIR)
        screen = result.get("reaction_screen")
        if (screen is None or tuple(screen["reactant_atom_counts"]) != tuple(REACTION_ATOM_COUNTS)
                or screen["bond_scale"] != BOND_SCALE or screen["frames_checked"] != result["frames_read"]):
            raise ValueError("Cache lacks the requested all-source-frame reaction screen. Set CACHE_DIR=None and rerun from RUN_DIR.")
        print("Using cached physics/settings:", result["settings"])
        print("Source:", result["files"])
except PossibleReactionError as exc:
    reaction_candidates = pd.DataFrame(exc.candidates)
    display(reaction_candidates)
    raise
frames = pd.DataFrame(result["frames"])
# Cached temperature may use an unknown velocity-bias convention; do not assess it.
frames = frames.drop(columns=["temperature_K"], errors="ignore")
thermo_source, thermo_coverage = None, None
if LOAD_THERMO and THERMO_CSV is None:
    thermo_directory = Path(result["files"]["dump"]).parent if result.get("files", {}).get("dump") else RUN_DIR
    thermo_source = thermo_io.discover_thermo_file(thermo_directory, thermo_file=THERMO_FILE)
    if thermo_source is not None:
        thermo = thermo_io.read_lammps_thermo(thermo_source, units=UNITS)
        frames, thermo_coverage = thermo_io.attach_thermo(frames, thermo, block=THERMO_BLOCK,
                                               dump_file=result["files"]["dump"])
        print("Thermo source:", thermo_source)
        print("Exact timestep matching:", thermo_coverage)
    else:
        print("No one-line thermo output found; temperature, pressure and energies remain unavailable.")
if THERMO_CSV is not None:
    thermo = pd.read_csv(THERMO_CSV)
    if "frame" not in thermo or thermo["frame"].duplicated().any():
        raise ValueError("Thermo CSV needs unique source frame indices")
    if not np.all(np.isfinite(thermo["frame"])) or not np.all(thermo["frame"] == np.floor(thermo["frame"])):
        raise ValueError("Thermo frame indices must be finite integers")
    if set(thermo.columns) & (set(frames.columns) - {"frame"}):
        raise ValueError("Thermo columns must not overwrite trajectory-derived columns")
    frames = frames.merge(thermo, on="frame", how="left", validate="one_to_one", sort=False)
species = np.asarray(result["species"])
print(f"{len(frames)} retained frames; {len(species)} molecules")
if SHOW_DETAILS:
    display(frames.head())
for notice in result.get("notices", []):
    print(notice)
print("Missing thermo observables:", [name for name in
      ("potential_energy_eV", "pressure_bar", "temperature_K") if name not in frames])

print("No target-size possible product detected in the checked source frames.")
print("This does not rule out other reactions or larger aggregates.")
sampling_rows = []
for start, end in WINDOWS:
    keep = select_frame_window(result, startframe=start, endframe=end)
    selected = frames.loc[keep]
    if len(selected) < 3:
        raise ValueError("Select at least three retained frames for these diagnostics")
    dt = float(np.diff(selected["time_ps"])[0])
    for duration in BLOCK_DURATIONS_PS:
        block_frames = max(3, int(np.ceil(duration/dt))+1)
        required_frames = MIN_BLOCKS*block_frames
        sampling_rows.append({
            "Source frames": f"{start}:{end if end is not None else result['frames_read']}",
            "Available time (ps)": float(selected["time_ps"].iloc[-1]-selected["time_ps"].iloc[0]),
            "Block duration (ps)": (block_frames-1)*dt,
            "Complete blocks": len(selected)//block_frames,
            "Required blocks": MIN_BLOCKS,
            "Required retained frames": required_frames,
            "Required time (ps)": (required_frames-1)*dt,
        })
sampling_table = pd.DataFrame(sampling_rows)
print("Sampling budget for the requested diffusion uncertainty:")
display(sampling_table)
print("Too few blocks means uncertainty is unresolved. It does NOT establish that the system is unequilibrated.")


## 2. Are the measured properties changing?
**Read the plots:** a continuing trend in density, volume, energy or temperature is a reason to investigate equilibration. A flat trace alone is insufficient; molecules may still rearrange slowly.

**Read the table:** it compares the earlier and later portions of each selected window using block averages. A *block* is a consecutive stretch of frames. This avoids pretending that every saved frame is independent.

Set `SCALAR_TOLERANCES` to the largest scientifically acceptable change in each quantity's own units. For example, `{"density_g_cm3": 0.01}` asks whether the change is resolved to within 0.01 g/cm3; this is an illustration, not a recommended tolerance. Without a tolerance, the notebook displays the evidence but makes no stability claim.


### Mathematics: whole-frame density and kinetic temperature
For $N$ atoms, cell matrix $H$ (Å), masses $m_i$ (amu) and velocities $\mathbf v_i$:
$$V=|\det H|,\qquad \rho=1.66053906660\,\frac{\sum_i m_i}{V}\quad[\mathrm{g/cm^3}],$$
$$K=\tfrac12\sum_i m_i|\mathbf v_i|^2,\qquad T=\frac{2K}{f_{\rm DOF}k_B}.$$
Kinetic energy is evaluated in ASE's internal mass/velocity units, giving eV; $k_B$ is in eV/K. `TEMPERATURE_DOF` supplies $f_{\rm DOF}$; `"3N-3"` sets $f_{\rm DOF}=3N-3$ and `None` omits temperature. **Current notebook calls do not subtract COM velocity.** Choosing $3N-3$ changes only the denominator; it does not remove drift. A drift-corrected calculation instead uses $\mathbf v_i-\mathbf V$, where $\mathbf V=\sum_i m_i\mathbf v_i/\sum_i m_i$, and requires `remove_com=True`. Constraints require their own DOF accounting. Notebook 08 omits this derived temperature and can accept a documented thermo temperature instead.

A displayed whole-run mean is $\bar x=\sum_{j=0}^{T-1}x_j/T$; it averages saved frames, not independent samples.


### Mathematics: scalar equivalence (separate from diffusion blocks)
For each observable, use only its actual recorded samples in the selected window; $T$ and $\Delta t$ refer to that observable, not necessarily dump sampling. Missing dump-aligned rows are not interpolated. Irregular observed spacing is reported as unavailable for this estimator.

For a selected scalar trace $x_0,\ldots,x_{T-1}$, this notebook tests two sample block sizes
$$b\in\{\max(3,\lfloor T/16\rfloor),\max(3,\lfloor T/8\rfloor)\}.$$
These are **not** `BLOCK_DURATIONS_PS`, and the scalar minimum of eight complete blocks is fixed independently of diffusion's `MIN_BLOCKS`. Compute $B=\lfloor T/b\rfloor$ block means $\bar x_q=b^{-1}\sum_{j=qb}^{(q+1)b-1}x_j$; discard $T-Bb$ tail samples. Split means into an earlier group of $\lceil B/2\rceil$ and a later group of $\lfloor B/2\rfloor$:
$$\Delta_x=\mathrm{mean}(\bar x_{\rm late})-\mathrm{mean}(\bar x_{\rm early}).$$
Resample each group independently with replacement, keeping its group size, for `BOOTSTRAP_RESAMPLES` draws. The 2.5/97.5 percentiles give $[L_x,U_x]$. With `SCALAR_TOLERANCES[name]` $\delta$ in the observable's units, require
$$-\delta\le L_x\quad\mathrm{and}\quad U_x\le\delta.$$
A CI merely containing zero does not pass equivalence. `None` leaves this claim unassessed.

Using the full selected trace, estimate autocorrelation by zero-padded FFT (no circular wrap):
$$C_k=\frac{\sum_{j=0}^{T-k-1}(x_j-\bar x)(x_{j+k}-\bar x)}{T-k},\quad\rho_k=C_k/C_0,$$
$$g=\max\left(1,1+2\sum_{k=1}^{K-1}\rho_k\right),\quad\tau_c=\frac{g\Delta t}{2},\quad N_{\rm eff}=T/g.$$
$K$ is the first nonpositive lag. The stationarity screen additionally needs a zero crossing, $B\ge8$, and $(b-1)\Delta t\ge10\tau_c$. A constant trace cannot establish independent sampling. These first-zero/ten-correlation-time rules are heuristics; slow unsampled modes can escape them. All reasons remain in `scalar_table`; the short summary displays a prioritized explanation.


In [ ]:
scalar_rows, scalar_details = [], {}
scalar_columns = [c for c in frames if c not in ("frame", "timestep", "time_ps", "thermo_time_ps")
                  and pd.api.types.is_numeric_dtype(frames[c])]
if not scalar_columns:
    print("No scalar observables available.")
else:
    fig, axes = plt.subplots(len(scalar_columns), 1, figsize=(10, 2.7*len(scalar_columns)), squeeze=False)
    for ax, name in zip(axes[:, 0], scalar_columns):
        observed = frames[name].notna()
        ax.plot(frames.loc[observed, "frame"], frames.loc[observed, name], lw=0.8)
        ax.set(xlabel="Source frame", ylabel=name)
    plt.tight_layout()
    plt.show()
for start, end in WINDOWS:
    keep = select_frame_window(result, startframe=start, endframe=end)
    chosen = frames.loc[keep]
    times = chosen["time_ps"].to_numpy()
    for name in scalar_columns:
        observed = chosen.loc[chosen[name].notna()]
        values = observed[name].to_numpy()
        times = observed["time_ps"].to_numpy()
        if len(values) < 3 or not np.isfinite(values).all():
            scalar_rows.append(dict(startframe=start, endframe=end, observable=name, status="missing/nonfinite samples"))
            continue
        spacing = np.diff(times)
        if np.any(spacing <= 0) or not np.allclose(spacing, spacing[0], rtol=1e-6, atol=1e-10):
            scalar_rows.append(dict(startframe=start, endframe=end, observable=name, status="irregular sample spacing"))
            continue
        print(f"{name}: {len(values)} observed samples, spacing {spacing[0]:g} ps in frames {start}:{end}")
        for block_frames in sorted(set([max(3, len(values)//16), max(3, len(values)//8)])):
            diagnostic = scalar_stationarity(values, times, block_frames=block_frames,
                tolerance=SCALAR_TOLERANCES.get(name), resamples=BOOTSTRAP_RESAMPLES, seed=SEED)
            key = f"{start}:{end}/{name}/{block_frames}"
            scalar_details[key] = diagnostic
            scalar_rows.append(dict(startframe=start, endframe=end, observable=name,
                block_frames=block_frames, **{k:v for k,v in diagnostic.items() if k != "block_means"}))
scalar_table = pd.DataFrame(scalar_rows)
scalar_summary_rows = []
for row in scalar_rows:
    reasons = row.get("reasons", [])
    if row["status"] == "irregular sample spacing":
        finding, action = "Irregular observable sampling", "Select a uniformly sampled segment; no interpolation was applied."
    elif row["status"] == "missing/nonfinite samples":
        finding, action = "Missing or nonfinite observable samples", "Check the thermo-to-frame mapping and source data."
    elif row["status"] == "supports stationarity":
        finding, action = "Earlier/later values agree within your tolerance", "Also inspect structure and other slow observables."
    elif any("Constant observable" in r for r in reasons):
        finding, action = "Constant trace cannot test equilibration", "Check whether this quantity is fixed by the simulation ensemble."
    elif row.get("blocks", 0) < 8:
        finding, action = "Too few temporal blocks", "Use a longer window; no stability conclusion yet."
    elif any("correlation times" in r for r in reasons):
        finding, action = "Samples remain too correlated at this block size", "Use longer blocks and enough trajectory to retain at least eight."
    elif any("not established within tolerance" in r for r in reasons):
        finding, action = "Agreement within your tolerance is not established", "Inspect the change interval and trace: drift or imprecision can cause this."
    elif row.get("tolerance") is None:
        finding, action = "No acceptable-change tolerance specified", "Choose a physical tolerance; a near-zero observed change alone is insufficient."
    else:
        finding, action = "Data unavailable or inconclusive", "Inspect missing values and detailed reasons."
    scalar_summary_rows.append({"Source frames": f"{row['startframe']}:{row['endframe']}",
        "Observable": row["observable"], "Frames per block": row.get("block_frames"),
        "Later minus earlier": row.get("late_minus_early"),
        "95% interval for change": row.get("difference_ci95"),
        "Acceptable change (+/-)": row.get("tolerance"), "Finding": finding, "Next step": action})
scalar_summary = pd.DataFrame(scalar_summary_rows)
display(scalar_summary)
if SHOW_DETAILS:
    display(scalar_table)


## 3. Do the molecules show sustained diffusion?
**Left: MSD.** This is the mean squared distance a molecular COM moves over a lag, averaged over molecules and starting frames. Diffusion needs a sustained rising straight region. A curve that flattens shows restricted motion over that timescale; fitting it does not establish a reliable D.

**Right: scaling exponent.** Around 1 means MSD grows roughly in proportion to lag. Around 0 means a plateau; around 2 indicates ballistic motion. The shaded band is a configurable screening range, not a universal physical boundary. Noise or an MSD offset can complicate this check.

`FIT_INTERVALS_PS` supplies several lag ranges for comparison. `BLOCK_DURATIONS_PS` supplies consecutive trajectory pieces used to estimate uncertainty. Neither setting changes which source frames you selected. A **candidate** means numerical screens passed for one fit, not that the system is equilibrated.


### Mathematics: molecular motion, MSD and correlations
For molecule $m$ of mass $M_m=\sum_{i\in m}m_i$, first reconstruct whole coordinates across periodic bonds:
$$\mathbf R_m=\frac{\sum_{i\in m}m_i\mathbf r_i}{M_m},\quad
\mathbf V_m=\frac{\sum_{i\in m}m_i\mathbf v_i}{M_m},\quad
\mathbf F_m=\sum_{i\in m}\mathbf F_i.$$
$\mathbf R_m$ is in Å, $\mathbf V_m$ in Å/ps, $\mathbf F_m$ in eV/Å. Whole-molecule COMs are temporally unwrapped by TOR, componentwise:
$$\mathbf d_j=\mathbf w_j-\mathbf w_{j-1},\quad
\Delta\mathbf r_j=\mathbf d_j-\mathbf L_j\left\lfloor\mathbf d_j/\mathbf L_j+\tfrac12\right\rfloor,\quad
\mathbf u_j=\mathbf u_{j-1}+\Delta\mathbf r_j.$$
$\mathbf w_j$ is the centered wrapped COM, $\mathbf L_j$ the destination box lengths, $\mathbf u_0=\mathbf w_0$. `analyze_run` updates this on every source frame before striding. This requires fixed-center orthorhombic periodic cells and sufficiently frequent source frames; no drift subtraction or finite-size correction is applied.

With $T$ selected samples and $M$ molecules of one species, lag $\tau_k=k\Delta t$:
$$\mathrm{MSD}(\tau_k)=\frac{\sum_{j=0}^{T-k-1}\sum_{m=1}^M|\mathbf u_{j+k,m}-\mathbf u_{j,m}|^2}{M(T-k)},\qquad N_{\rm origins}=T-k,$$
$$C_X(\tau_k)=\frac{\sum_{j=0}^{T-k-1}\sum_m\mathbf X_{j,m}\cdot\mathbf X_{j+k,m}}{M(T-k)}.$$
$X=V$ gives VACF in (Å/ps)²; $X=F$ gives force correlation in (eV/Å)². No temporal mean is subtracted. Plots use $C_X(\tau)/C_X(0)$ when $C_X(0)>0$. Overlapping origins are correlated. Notebook 08 uses the sampled lag grid and fit equations in the next Markdown cell.


### Mathematics: diffusion fits, bootstrap and rejection rules
The molecular MSD above uses all $M(T-k)$ molecule/origin comparisons at each sampled lag. The lag grid is capped at 100 positive integer frame lags (evenly spaced then integer-truncated/deduplicated) plus zero, up to $\min(T-1,\lfloor\max\tau_b/\Delta t\rfloor)$. It is **not** every lag for long windows. No ordinary regression standard error is interpreted as physical uncertainty.

For fit interval $I$, the pooled fit is
$$ (c,s)=\arg\min_{c,s}\sum_{k:\tau_k\in I}[\mathrm{MSD}_k-c-s\tau_k]^2,\qquad D_{\rm pooled}=s/6.$$
Independently fit the log data for $\alpha_I$; the plotted local slope instead approximates $d\log(\mathrm{MSD})/d\log\tau$ via `np.gradient` (centered three-point finite differences on the nonuniform log-lag grid, one-sided endpoints). Both omit nonpositive lags/MSD; a nonpositive MSD in a fitted interval makes its exponent unavailable.

For each complete block $q=1,\ldots,B$, compute its own MSD with origins wholly inside the block and fit $D_q=s_q/6$. The incomplete tail $T-Bb$ is excluded from blocks but included in the pooled curve. Let $\bar D=B^{-1}\sum_qD_q$. A bootstrap draw samples $B$ block indices independently with replacement:
$$\bar D^{*(r)}=\frac1B\sum_{j=1}^{B}D_{I_j^{(r)}},\qquad
[L,U]=[Q_{0.025}(\bar D^*),Q_{0.975}(\bar D^*)].$$
$Q_p$ is NumPy's linearly interpolated empirical percentile. Resample all molecules together within a block; never treat molecules or overlapping origins as independent replicates. Reported `D_m2_s` and its CI are $10^{-8}\bar D$ and $10^{-8}[L,U]$. Intervals assume sufficiently independent, stationary blocks.

In addition to sufficient blocks and fitted exponent bounds, a candidate requires $\bar D>0$, $L>0$, and
$$\frac{U-L}{2}\le\eta|\bar D|,$$
$$\frac{|\bar D_{\rm early}-\bar D_{\rm late}|}{\max(|\bar D|,10^{-30})}\le\epsilon,\qquad
|D_{\rm pooled}-\bar D|\le\epsilon\max(|\bar D|,10^{-30}).$$
Early diffusion blocks are the first $\lfloor B/2\rfloor$; the rest are late. Across at least three distinct fit intervals, also require
$$\frac{\max_I D_{{\rm pooled},I}-\min_I D_{{\rm pooled},I}}
{\max(|\mathrm{mean}_I D_{{\rm pooled},I}|,10^{-30})}\le\epsilon.$$
$\epsilon=$ `RELATIVE_TOLERANCE`, $\eta=$ `MAX_RELATIVE_CI`; $10^{-30}$ only protects division near zero in Å²/ps. Candidate status is per row, not a combined equilibrium decision or a block-size convergence test. Cross-block-size/window agreement still requires inspection.


In [ ]:
diffusion_rows, diffusion_details = [], {}
if result["com_angstrom"] is None:
    print("Diffusion unavailable: see the cell/unwrapping notices above.")
else:
    for start, end in WINDOWS:
        keep = select_frame_window(result, startframe=start, endframe=end)
        times = frames.loc[keep, "time_ps"].to_numpy()
        dt = np.diff(times)[0]
        positions = np.asarray(result["com_angstrom"])[keep]
        fig, axes = plt.subplots(1, 2, figsize=(11, 4))
        for label in sorted(set(species)):
            curves = []
            for duration in BLOCK_DURATIONS_PS:
                block_frames = max(3, int(np.ceil(duration/dt))+1)
                diagnostic = diffusion_diagnostics(positions[:, species == label], times,
                    fit_intervals_ps=FIT_INTERVALS_PS, block_frames=block_frames,
                    min_blocks=MIN_BLOCKS, resamples=BOOTSTRAP_RESAMPLES, seed=SEED,
                    alpha_bounds=ALPHA_BOUNDS, relative_tolerance=RELATIVE_TOLERANCE,
                    max_relative_ci=MAX_RELATIVE_CI)
                key = f"{start}:{end}/{label}/{block_frames}"
                diffusion_details[key] = diagnostic
                curves.append(diagnostic)
                for fit in diagnostic["fits"]:
                    diffusion_rows.append(dict(startframe=start, endframe=end, species=label,
                        block_duration_ps=(block_frames-1)*dt, blocks=diagnostic["blocks"],
                        discarded_samples=diagnostic["discarded_block_samples"],
                        **{k:v for k,v in fit.items() if k != "block_D_angstrom2_ps"}))
            curve = curves[0]
            lag, msd = curve["lag_ps"], curve["msd_angstrom2"]
            axes[0].plot(lag, msd, label=label)
            positive = (lag > 0) & (msd > 0)
            if positive.sum() >= 3:
                alpha = np.gradient(np.log(msd[positive]), np.log(lag[positive]))
                axes[1].plot(lag[positive], alpha, label=label)
        axes[0].set(xlabel="Lag (ps)", ylabel="Molecular COM MSD (Angstrom^2)")
        axes[1].axhspan(*ALPHA_BOUNDS, alpha=.15, color="green")
        axes[1].axhline(1, color="black", ls="--")
        axes[1].set(xlabel="Lag (ps)", ylabel="Local log slope (noisy diagnostic)")
        for ax in axes:
            ax.legend()
            ax.grid(alpha=.2)
        fig.suptitle(f"Source frames [{start}, {end if end is not None else result['frames_read']})")
        plt.tight_layout()
        plt.show()
diffusion_table = pd.DataFrame(diffusion_rows)
diffusion_summary_rows = []
if not diffusion_table.empty:
    for (start, end, label), group in diffusion_table.groupby(["startframe", "endframe", "species"], dropna=False):
        reasons = sorted({reason for values in group["reasons"] for reason in values})
        passed = int((group["status"] == "candidate").sum())
        if passed == len(group):
            finding = "All requested fit screens pass; candidate D estimates"
            action = "Compare uncertainty across block sizes, later windows and independent runs; assess stability separately."
        elif passed:
            finding = f"Mixed evidence: {passed} of {len(group)} fits pass"
            action = "Do not select only the passing fit. Inspect which lag ranges or block sizes disagree."
        elif all(status == "insufficient sampling" for status in group["status"]):
            finding = "Requested lag ranges cannot be evaluated"
            action = "Check the sampling budget. Supply longer data or physically justified lag ranges."
        elif any("MSD scaling" in r for r in reasons):
            finding = "A sustained diffusive regime is not established"
            action = "Inspect MSD flattening/curvature and longer lags; D is not reportable from these screens."
        elif any("blocks" in r.lower() or "block span" in r.lower() for r in reasons):
            finding = "Not enough temporal sampling for uncertainty"
            action = "Use more trajectory and adequate block lengths; more time origins do not fix this."
        else:
            finding = "Diffusion estimates are not consistent or precise enough"
            action = "Inspect temporal drift, fit sensitivity and uncertainty before reporting D."
        diffusion_summary_rows.append({"Source frames": f"{int(start)}:{'end' if pd.isna(end) else int(end)}",
            "Species": label, "Finding": finding, "Why": "; ".join(reasons) if reasons else "All requested numerical screens passed",
            "Next step": action})
diffusion_summary = pd.DataFrame(diffusion_summary_rows)
display(diffusion_summary)
if SHOW_DETAILS:
    display(diffusion_table)


### Diffusion numbers and their uncertainty
The compact table shows the mean block D and its conditional 95% interval. **An estimate marked unsupported is diagnostic only; do not report it as a measured diffusion coefficient.** The interval assumes blocks are sufficiently independent.

Compare estimates across lag ranges and block lengths. Longer blocks should not cause large unexplained shifts or ever-growing uncertainty. `SHOW_DETAILS=True` adds interval plots and consecutive block estimates; all original arrays remain available in `diffusion_details`.


In [ ]:
if not diffusion_table.empty and "D_m2_s" in diffusion_table:
    columns = ["startframe", "endframe", "species", "fit_start_ps", "fit_end_ps",
               "block_duration_ps", "D_m2_s", "ci95_m2_s", "status"]
    display(diffusion_table[columns].rename(columns={
        "D_m2_s": "Estimated D (m2/s)", "ci95_m2_s": "Conditional 95% interval",
        "status": "Fit screen (not equilibrium)"}))
else:
    print("No diffusion estimate can be evaluated for these lag intervals.")
if SHOW_DETAILS:
    if not diffusion_table.empty and "D_m2_s" in diffusion_table:
        for label in sorted(set(diffusion_table["species"])):
            rows = diffusion_table[diffusion_table["species"] == label].dropna(subset=["D_m2_s"])
            if rows.empty:
                continue
            fig, ax = plt.subplots(figsize=(10, max(3, .35*len(rows))))
            for y, (_, row) in enumerate(rows.iterrows()):
                lo, hi = row["ci95_m2_s"]
                ax.plot([lo, hi], [y, y], color="C0")
                ax.plot(row["D_m2_s"], y, "o", color="C0")
            ax.set_yticks(range(len(rows)), [f"frames {r.startframe}:{r.endframe}, lag {r.fit_start_ps:g}-{r.fit_end_ps:g}, block {r.block_duration_ps:g} ps"
                                          for r in rows.itertuples()])
            ax.set(xlabel="D (m^2/s); conditional 95% block interval", title=label)
            plt.tight_layout()
            plt.show()
    for key, diagnostic in diffusion_details.items():
        fits_with_blocks = [f for f in diagnostic["fits"] if "block_D_angstrom2_ps" in f]
        if fits_with_blocks:
            fit = fits_with_blocks[0]
            fig, ax = plt.subplots(figsize=(8, 2.5))
            ax.plot(np.arange(diagnostic["blocks"]), np.array(fit["block_D_angstrom2_ps"])*1e-8, "o-")
            ax.set(xlabel="Consecutive temporal block", ylabel="D (m^2/s)",
                   title=f"{key}; lag {fit['fit_start_ps']:g}-{fit['fit_end_ps']:g} ps")
            plt.tight_layout()
            plt.show()


## 4. Is molecular structure still changing?
Each plot compares the COM radial distribution g(r) in four consecutive sections of your window. Peaks show preferred molecular separations.

**Changing peak positions or heights:** structural evolution may still be occurring. **Overlapping curves:** supports similar structure at this resolution, but does not establish equilibrium. There are no uncertainty bands here. Inspect slower observables such as orientation or conformation when relevant.


### Mathematics: molecular RDF and nearest neighbors
Use instantaneous whole-molecule **wrapped** COMs, not accumulated TOR positions. For orthorhombic box lengths $\mathbf L_t$:
$$\mathbf d_{ij}=\mathbf R_j-\mathbf R_i-\mathbf L_t\left\lfloor(\mathbf R_j-\mathbf R_i)/\mathbf L_t+\tfrac12\right\rfloor.$$
For radial bin $[r_l,r_{l+1})$, let $n_{tl}$ count pair distances in that bin and $V_l=4\pi(r_{l+1}^3-r_l^3)/3$. Then
$$g_l=\frac1{T_s}\sum_{t\in\mathrm{sampled}}\frac{n_{tl}\prod_\alpha L_{t\alpha}}{P V_l},\quad
P=\begin{cases}M_A M_B&A\ne B,\\M_A(M_A-1)/2&A=B.\end{cases}$$
Like-species pairs are counted once, self-pairs excluded. Bin centers are $(r_l+r_{l+1})/2$. `RDF_BINS` sets the number of equal-width bins where configurable; notebook 05 fixes 64 bins and ends at $0.999\min_{t,\alpha}L_{t\alpha}/2$. Notebook 08 ends at half the shortest selected box edge, divides the window into four chronological sections, and samples every `STRUCTURE_STRIDE`-th retained frame **within each section**. These are descriptive curves without uncertainty bands.

Nearest-neighbor distances are $d_{ti}^{\rm nearest}=\min_j\|\mathbf d_{ij}\|$ (exclude self); the reported mean averages over central molecules and frames. Nearest distances are not limited by the RDF bin range.


In [ ]:
if result.get("wrapped_com_angstrom") is not None:
    labels = sorted(set(species))
    for start, end in WINDOWS:
        keep = select_frame_window(result, startframe=start, endframe=end)
        indices = np.flatnonzero(keep)
        boxes = np.asarray(result["box_lengths_angstrom"])
        positions = np.asarray(result["wrapped_com_angstrom"])
        edges = np.linspace(0, boxes[indices].min()/2, RDF_BINS+1)
        for ia, a in enumerate(labels):
            for b in labels[ia:]:
                if a == b and np.count_nonzero(species == a) < 2:
                    continue
                fig, ax = plt.subplots(figsize=(7, 3))
                for section, chunk in enumerate(np.array_split(indices, 4)):
                    if not len(chunk):
                        continue
                    chosen = chunk[::STRUCTURE_STRIDE]
                    rdf = get_molecular_com_rdf(positions[chosen], boxes[chosen], species,
                                               species_a=a, species_b=b, edges=edges)
                    ax.plot(rdf["r_angstrom"], rdf["g_r"], label=f"section {section+1}")
                ax.set(xlabel="COM separation (Angstrom)", ylabel="g(r)", title=f"{a} / {b}; frames {start}:{end}")
                ax.legend()
                plt.tight_layout()
                plt.show()
else:
    print("COM RDF unavailable for this cell convention.")


## 5. What can I conclude?
- **Stable scalar/structural behavior + consistent diffusive motion:** supports further validation of a candidate D; compare later windows and independent runs.
- **Stable properties + flat MSD:** the observed state may be stable, but diffusion is unresolved on this timescale.
- **Changing properties or structure:** equilibrium estimates from that window are not yet supported.
- **Too few blocks / unavailable data:** insufficient evidence, not a diagnosis of nonequilibrium.

The notebook does not combine these into a misleading “equilibrated: yes/no.” Use the findings above to decide which observation needs more sampling or investigation. Your full detailed tables remain in `scalar_table` and `diffusion_table`.


In [ ]:
print("Assessment complete. Read the findings in sections 2 and 3 alongside the structure plots in section 4.")
print("No automatic equilibrium certification or single preferred diffusion coefficient is produced.")
if WRITE_FILES:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=False)  # choose a new directory for each assessment
    scalar_table.to_csv(OUTPUT_DIR / "scalar_stationarity.csv", index=False)
    diffusion_table.to_csv(OUTPUT_DIR / "diffusion_diagnostics.csv", index=False)
    scalar_summary.to_csv(OUTPUT_DIR / "scalar_findings.csv", index=False)
    diffusion_summary.to_csv(OUTPUT_DIR / "diffusion_findings.csv", index=False)
    sampling_table.to_csv(OUTPUT_DIR / "sampling_budget.csv", index=False)
    frames.to_csv(OUTPUT_DIR / "frame_observables.csv", index=False)
    settings = dict(run_dir=str(RUN_DIR), cache_dir=str(CACHE_DIR), units=UNITS,
        thermo_source=str(thermo_source), thermo_coverage=thermo_coverage, thermo_block=THERMO_BLOCK,
        timestep_ps=TIMESTEP_PS, source_files=result["files"], source_settings=result["settings"],
        windows=WINDOWS, frame_stride=FRAME_STRIDE, max_frames=MAX_FRAMES,
        fit_intervals_ps=FIT_INTERVALS_PS, block_durations_ps=BLOCK_DURATIONS_PS,
        reaction_atom_counts=REACTION_ATOM_COUNTS, bond_scale=BOND_SCALE,
        min_blocks=MIN_BLOCKS, resamples=BOOTSTRAP_RESAMPLES, seed=SEED,
        alpha_bounds=ALPHA_BOUNDS, relative_tolerance=RELATIVE_TOLERANCE,
        max_relative_ci=MAX_RELATIVE_CI, scalar_tolerances=SCALAR_TOLERANCES)
    (OUTPUT_DIR / "settings.json").write_text(json.dumps(settings, indent=2, default=str), encoding="utf-8")
    print("Wrote", OUTPUT_DIR.resolve())


## Methods and limitations
Package functions: `trajectory_analysis.analyze_run` (stream/reconstruct), `select_frame_window` (source indices), `diffusion_diagnostics.diffusion_diagnostics` (fits/block uncertainty), `scalar_stationarity` (equivalence screen), and `stability.get_molecular_com_rdf` (instantaneous structure). No changes to pair selection or visualization bonds.

- [Bullerjahn et al. (2023), NPT unwrapping](https://doi.org/10.1021/acs.jctc.3c00308): TOR and molecular COM conventions.
- [Bullerjahn et al. (2020), diffusion estimation](https://arxiv.org/abs/2003.09193): correlated MSD errors matter. This notebook uses temporal-block resampling, **not** that paper's optimal estimator.
- [Chodera (2016), equilibration detection](https://doi.org/10.1021/acs.jctc.5b00784): equilibration assessment depends on the observable and correlation times. This notebook does **not** implement automated burn-in selection.

Independent replicas and system-size checks remain important. NPT/barostat artifacts, unresolved boundary crossings, reactions changing molecular identity, and sampling slower than the available record can invalidate a numerical candidate.


<details>
<summary>Optional: equations, assumptions and implementation details</summary>

## Method details (optional)

## Load once; reuse while changing analysis settings
`analyze_run` streams the dump and reconstructs all molecular COMs. TOR unwrapping occurs before striding; pair screening is disabled because it is unnecessary here. No GPU/model is used.

Automatic directory discovery finds the input, data and dump files; ambiguous matches require an override. The analysis currently supports fixed molecular connectivity and fixed-center orthorhombic periodic boxes for TOR. Saved frames must resolve boundary crossings.

We deliberately do not request the current atomic temperature implementation's `3N-3` option: it does not itself remove COM drift. Supply the actual simulation temperature through the optional thermo CSV if desired. Pressure and potential energy are read from a discovered LAMMPS output when available; they are not inferred from COM motion or atomic forces.


## Possible reactions
Every available source frame is checked for a connected component with `sum(REACTION_ATOM_COUNTS)` atoms (34 for current CPD/MAL). The notebook displays the first candidate and stops, even outside selected statistics windows. Initial-frame products also stop analysis. Old caches without this screen are rejected.

This is a distance-based candidate flag, not proof of the intended product. Inspect atom IDs, formula, new bonds and persistence. Use `reaction_detection.scan_possible_products(iter_lammps_frames(...), reactant_atom_counts=(11, 23))` to collect all occurrences. Larger aggregates or other reactions can escape this exact-size rule. A preview checks only frames actually read.

After inspection, analyze an explicitly truncated pre-reaction segment or develop species-aware post-reaction statistics. Do not mix reactant and product motion under the original species labels. Rebuilding connectivity adds cost.


## Scalar stationarity and block-size sensitivity
Compare the earlier and later groups of **nonoverlapping temporal block means**. The 95% bootstrap interval for their difference must lie entirely within your absolute tolerance; simply including zero is insufficient.

The FFT autocorrelation supplies a first-zero correlation-time heuristic. Blocks must span at least ten estimated correlation times, with at least eight blocks. Compare two block sizes. Slow unsampled modes, constant traces, missing data, and insufficient blocks remain inconclusive. Density and volume are related observables, not independent confirmations.

This is an equivalence screen, not automatic detection of a unique equilibration time. Inspect the full trace before choosing later source-frame windows.


## Diffusive regime, uncertainty and time-window dependence
Every species uses **all its molecules**. Plot MSD and the apparent exponent α=d log(MSD)/d log(lag): diffusion requires sustained near-linear growth; plateaus approach α=0. Positive MSD alone does not establish diffusion.

For each explicit interval, fit MSD = intercept + 6D·lag. Independently fit each complete temporal block and bootstrap the **block estimates**, keeping all molecules together. Report the mean block D and a conditional 95% interval, not an ordinary regression error over correlated MSD points. No displacement crosses a block boundary. The full-window curve uses all samples; block statistics disclose any dropped tail.

At least eight blocks, each spanning five maximum fit lags, are required by default. This is deliberately demanding. Too little trajectory produces an inconclusive result, not a fabricated precise D. Compare longer blocks: the CI and estimate should stabilize. Block independence is an assumption, not guaranteed by these length rules.

The configurable screening thresholds are heuristics. A substantial MSD intercept can make the exponent check conservative. Lag sampling is capped at 100 points for computational cost. The reported D is a finite-system value; no hydrodynamic finite-size correction is applied.


## Inspect block estimates and fit sensitivity
Horizontal bars below are conditional bootstrap intervals. The second panel shows consecutive block estimates for one fit interval: visible drift or long runs of correlated values undermine the independence assumption. A small CI alone is not sufficient.

Change WINDOWS to compare explicit later windows, rerun the analysis cells, and check whether the estimates change. Do not select a window only because it produces a preferred D.


## Structural stability
Compare species COM RDFs in four chronological sections of each selected window, using **instantaneous wrapped geometry**, not accumulated TOR coordinates. Changes in peak positions or amplitudes signal structural evolution even when density looks stable. Curves are descriptive, without independent-sample error bars; overlap does not prove equilibrium. Add system-specific observables (orientation, association, conformations) when they relax more slowly.


## Decision and next simulation
A **candidate** row is not a certified reliable result. To report D:
1. Establish stationary scalar and relevant structural observables in the same explicit window; disclose missing observables.
2. Find sustained diffusive MSD scaling and consistent D across lag intervals.
3. Require enough long blocks; compare at least two block sizes and inspect their time series.
4. Confirm agreement across later source windows and preferably independent simulations.

If MSD remains flat, extend sampling or report that diffusion is unresolved on this timescale. If scalar/structural properties drift, continue equilibration before treating the trajectory as stationary. A system may have stationary observables without enough diffusive motion to estimate D.

The table below summarizes numerical screens only. It intentionally does not label the system “equilibrated.”


</details>
